# 03_2 Selected Best Pipeline: 2.5D Physical ICH-Prior Transfer

Pick chosen from the research plan: external ICH teacher -> soft hemorrhage prior on PHE-SICH -> single 2.5D physical PHE student. No experiment matrix and no baseline-driven methodology drift.


## 1. Chosen method contract

Chosen configuration for `03_2`:

- Method: teacher-student external ICH-prior transfer.
- Input: 2.5D physical stack, offsets `[-6, -3, 0, +3, +6]` mm.
- Channels: 3 CT windows per slice, 15 channels total.
- Teacher source: Seg-CQ500 + INSTANCE if present + PhysioNet if present after DUA.
- Student target: PHE-SICH SubdatasetA NIfTI, locked 19c train/val/test.
- Heads/losses: PHE segmentation, prior alignment, area regression, aleatoric uncertainty.
- Metric unit: patient/volume; slice Dice is diagnostic only.

Previous notebooks are not used as method templates here. They can be evaluated later as external baselines with the same labels and metrics.


In [ ]:
from pathlib import Path
from dataclasses import dataclass, asdict


@dataclass
class ExperimentConfig:
    project_root: Path = Path.cwd()
    seed: int = 42
    image_size: int = 384
    smoke_image_size: int = 256
    n_folds: int = 5
    outer_fold: int = 0
    val_fraction: float = 0.125
    label_fractions: tuple = (0.10, 0.25, 0.50, 1.00)
    windows: tuple = (
        ("brain", -20.0, 100.0),
        ("blood", -50.0, 150.0),
        ("wide", 0.0, 600.0),
    )
    physical_offsets_mm: tuple = (-6.0, -3.0, 0.0, 3.0, 6.0)
    run_heavy_training: bool = False
    run_smoke_test: bool = True
    max_smoke_subjects: int = 2
    selected_mode: str = "25d"
    selected_label_fraction: float = 1.00
    teacher_epochs: int = 80
    student_epochs: int = 80
    teacher_batch_size: int = 8
    student_batch_size: int = 8
    # --- Optimization parameters (NEW) ---
    patience: int = 12
    min_lr: float = 1e-6
    warmup_epochs: int = 5
    grad_clip_norm: float = 1.0
    use_augmentation: bool = True
    aug_prob: float = 0.5
    teacher_val_fraction: float = 0.15


CFG = ExperimentConfig()
PROJECT_ROOT = CFG.project_root.resolve()

PHE_ROOT = PROJECT_ROOT / "PHE-SICH-CT-IDS" / "SubdatasetA_NIFIT" / "NIFIT"
PHE_IMAGE_DIR = PHE_ROOT / "set"
PHE_MASK_DIR = PHE_ROOT / "label"

SEG_CQ500_ROOT = PROJECT_ROOT / "Seg-CQ500" / "Seg-CQ500" / "data" / "volumes"
INSTANCE_ROOT = PROJECT_ROOT / "Instance2022" / "train_2"
PHYSIONET_ROOT = PROJECT_ROOT / "PhysioNet_CT_ICH"  # optional, restricted access

OUTPUT_ROOT = PROJECT_ROOT / "outputs_03_2_selected_25d_prior_transfer"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
FIG_DIR = OUTPUT_ROOT / "figures"
TABLE_DIR = OUTPUT_ROOT / "tables"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
PRED_DIR = OUTPUT_ROOT / "predictions"
PRIOR_DIR = OUTPUT_ROOT / "teacher_priors"
LOG_DIR = OUTPUT_ROOT / "logs"

for path in [OUTPUT_ROOT, MANIFEST_DIR, FIG_DIR, TABLE_DIR, CHECKPOINT_DIR, PRED_DIR, PRIOR_DIR, LOG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Auto-detect optimal num_workers
import os as _os
NUM_WORKERS = min(4, _os.cpu_count() or 0)

print("Project root:", PROJECT_ROOT)
print("Output root :", OUTPUT_ROOT)
print("Heavy training:", CFG.run_heavy_training)
print("Selected mode:", CFG.selected_mode)
print("Selected label fraction:", CFG.selected_label_fraction)
print("Num workers:", NUM_WORKERS)

In [ ]:
import os
import json
import math
import random
import time
from collections import defaultdict
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

try:
    import nibabel as nib
    NIB_AVAILABLE = True
except Exception as exc:
    nib = None
    NIB_AVAILABLE = False
    print("nibabel is not available:", exc)

try:
    from scipy import ndimage
    from scipy.stats import pearsonr, spearmanr, wilcoxon
    SCIPY_AVAILABLE = True
except Exception as exc:
    ndimage = None
    pearsonr = spearmanr = wilcoxon = None
    SCIPY_AVAILABLE = False
    print("scipy is not available:", exc)

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    TORCH_AVAILABLE = True
except Exception as exc:
    torch = nn = F = Dataset = DataLoader = None
    TORCH_AVAILABLE = False
    print("torch is not available. Training cells will be skipped:", exc)


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if TORCH_AVAILABLE:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


set_seed(CFG.seed)
DEVICE = "cuda" if TORCH_AVAILABLE and torch.cuda.is_available() else "cpu"
print({"nibabel": NIB_AVAILABLE, "scipy": SCIPY_AVAILABLE, "torch": TORCH_AVAILABLE, "device": DEVICE})


## 2. Audit dữ liệu local

Cell này chỉ kiểm tra những bộ dữ liệu đang có trong workspace. Nếu PhysioNet chưa xuất hiện thì notebook vẫn chạy với track lõi hiện có: PHE-SICH + Seg-CQ500 + INSTANCE.


In [ ]:
def count_nifti_files(root: Path) -> int:
    if not root.exists():
        return 0
    return sum(1 for p in root.rglob("*") if p.is_file() and (p.name.endswith(".nii") or p.name.endswith(".nii.gz")))


def dataset_audit_table() -> pd.DataFrame:
    rows = [
        {
            "dataset": "PHE-SICH-CT-IDS",
            "role": "target PHE",
            "root": str(PHE_ROOT),
            "exists": PHE_ROOT.exists(),
            "nifti_files": count_nifti_files(PHE_ROOT),
            "notes": "Use SubdatasetA_NIFIT/NIFIT/set + label",
        },
        {
            "dataset": "Seg-CQ500",
            "role": "source ICH",
            "root": str(SEG_CQ500_ROOT),
            "exists": SEG_CQ500_ROOT.exists(),
            "nifti_files": count_nifti_files(SEG_CQ500_ROOT),
            "notes": "Pair CT.nii with ICH_mask.nii.gz",
        },
        {
            "dataset": "INSTANCE 2022",
            "role": "source ICH optional extension",
            "root": str(INSTANCE_ROOT),
            "exists": INSTANCE_ROOT.exists(),
            "nifti_files": count_nifti_files(INSTANCE_ROOT),
            "notes": "Use train_2/data + train_2/label only",
        },
        {
            "dataset": "PhysioNet CT-ICH",
            "role": "source ICH optional restricted",
            "root": str(PHYSIONET_ROOT),
            "exists": PHYSIONET_ROOT.exists(),
            "nifti_files": count_nifti_files(PHYSIONET_ROOT),
            "notes": "Expected after DUA; common folders: ct_scans, masks",
        },
    ]
    return pd.DataFrame(rows)


audit_df = dataset_audit_table()
display(audit_df)
audit_df.to_csv(TABLE_DIR / "dataset_audit.csv", index=False, encoding="utf-8")


## 3. NIfTI, windowing và 2.5D theo độ sâu vật lý

Các helper dưới đây giữ đúng tinh thần plan: chuẩn hóa cường độ bằng nhiều cửa sổ CT, tạo 2D stack 3 kênh, và tạo 2.5D stack bằng offset vật lý theo mm thay vì lấy lát `z-2:z+2` một cách ngây thơ.


In [ ]:
def is_nifti(path: Path) -> bool:
    name = path.name.lower()
    return name.endswith(".nii") or name.endswith(".nii.gz")


def strip_nii_suffix(path_or_name) -> str:
    name = Path(path_or_name).name
    if name.endswith(".nii.gz"):
        return name[:-7]
    if name.endswith(".nii"):
        return name[:-4]
    return Path(name).stem


def load_nifti(path: Path, canonical: bool = True):
    if not NIB_AVAILABLE:
        raise ImportError("Install nibabel to read NIfTI files: pip install nibabel")
    img = nib.load(str(path))
    if canonical:
        img = nib.as_closest_canonical(img)
    data = np.asanyarray(img.dataobj)
    spacing = tuple(float(v) for v in img.header.get_zooms()[:3])
    return data, spacing, img.affine


def nifti_header_row(path: Path) -> Dict:
    if not NIB_AVAILABLE:
        return {"shape": None, "spacing": None, "n_slices": None}
    img = nib.load(str(path))
    shape = tuple(int(v) for v in img.shape[:3])
    spacing = tuple(float(v) for v in img.header.get_zooms()[:3])
    return {
        "shape": shape,
        "spacing": spacing,
        "spacing_x": spacing[0],
        "spacing_y": spacing[1],
        "spacing_z": spacing[2],
        "n_slices": shape[2] if len(shape) >= 3 else 1,
    }


def mask_stats(mask_path: Path, spacing: Tuple[float, float, float]) -> Dict:
    if not mask_path or not Path(mask_path).exists() or not NIB_AVAILABLE:
        return {"mask_voxels": 0, "mask_volume_ml": 0.0, "positive_slices": 0}
    mask = np.asanyarray(nib.load(str(mask_path)).dataobj) > 0
    voxel_volume_mm3 = float(np.prod(spacing))
    per_slice = mask.reshape((-1, mask.shape[-1])).sum(axis=0)
    return {
        "mask_voxels": int(mask.sum()),
        "mask_volume_ml": float(mask.sum() * voxel_volume_mm3 / 1000.0),
        "positive_slices": int((per_slice > 0).sum()),
    }


def window_ct(image: np.ndarray, low: float, high: float) -> np.ndarray:
    image = image.astype(np.float32, copy=False)
    clipped = np.clip(image, low, high)
    return ((clipped - low) / max(high - low, 1e-6)).astype(np.float32)


def make_window_stack(slice_2d: np.ndarray, windows=CFG.windows) -> np.ndarray:
    channels = [window_ct(slice_2d, low, high) for _, low, high in windows]
    return np.stack(channels, axis=0)


def resize_2d(array: np.ndarray, out_size: int, order: int = 1) -> np.ndarray:
    if array.shape[-2:] == (out_size, out_size):
        return array
    if not SCIPY_AVAILABLE:
        raise ImportError("Install scipy for resizing or provide pre-resized caches.")
    zoom_y = out_size / array.shape[-2]
    zoom_x = out_size / array.shape[-1]
    if array.ndim == 2:
        return ndimage.zoom(array, (zoom_y, zoom_x), order=order)
    if array.ndim == 3:
        return np.stack([ndimage.zoom(ch, (zoom_y, zoom_x), order=order) for ch in array], axis=0)
    raise ValueError(f"Unsupported resize shape: {array.shape}")


def physical_neighbor_indices(center_idx: int, n_slices: int, spacing_z: float, offsets_mm=CFG.physical_offsets_mm) -> List[int]:
    z_positions = np.arange(n_slices, dtype=np.float32) * float(spacing_z)
    center_z = z_positions[int(center_idx)]
    selected = []
    for offset in offsets_mm:
        target_z = center_z + float(offset)
        selected.append(int(np.argmin(np.abs(z_positions - target_z))))
    return selected


def make_2d_stack(volume: np.ndarray, center_idx: int, image_size: int = CFG.image_size) -> np.ndarray:
    slice_2d = np.asarray(volume[:, :, center_idx])
    stack = make_window_stack(slice_2d)
    return resize_2d(stack, image_size, order=1).astype(np.float32)


def make_25d_stack(volume: np.ndarray, center_idx: int, spacing_z: float, image_size: int = CFG.image_size) -> np.ndarray:
    indices = physical_neighbor_indices(center_idx, volume.shape[2], spacing_z)
    stacks = [make_window_stack(np.asarray(volume[:, :, z])) for z in indices]
    stack = np.concatenate(stacks, axis=0)
    return resize_2d(stack, image_size, order=1).astype(np.float32)


def resize_mask(mask_2d: np.ndarray, image_size: int = CFG.image_size) -> np.ndarray:
    mask = resize_2d(mask_2d.astype(np.float32), image_size, order=0)
    return (mask > 0.5).astype(np.float32)


## 4. Build manifest toàn cục

Manifest giữ `spacing`, `n_slices`, `task`, `role`, `license` và thống kê mask để phục vụ split patient-level, low-label và đánh giá thể tích. Đây là deliverable tái lập quan trọng nhất trước khi train.


In [ ]:
def build_phe_manifest() -> pd.DataFrame:
    rows = []
    if not PHE_IMAGE_DIR.exists() or not PHE_MASK_DIR.exists():
        return pd.DataFrame()
    images = {strip_nii_suffix(p): p for p in PHE_IMAGE_DIR.iterdir() if p.is_file() and is_nifti(p)}
    masks = {strip_nii_suffix(p): p for p in PHE_MASK_DIR.iterdir() if p.is_file() and is_nifti(p)}
    for scan_id, img_path in sorted(images.items()):
        mask_path = masks.get(scan_id)
        header = nifti_header_row(img_path)
        stats = mask_stats(mask_path, header["spacing"]) if mask_path else {}
        rows.append(
            {
                "dataset": "PHE-SICH-CT-IDS",
                "role": "target",
                "task": "phe_segmentation",
                "patient_id": f"phe_{scan_id}",
                "scan_id": scan_id,
                "img_path": str(img_path),
                "mask_path": str(mask_path) if mask_path else "",
                "source_license": "CC BY 4.0",
                "has_phe": bool(stats.get("mask_voxels", 0) > 0),
                "has_ich": "",
                **header,
                **stats,
            }
        )
    return pd.DataFrame(rows)


def build_seg_cq500_manifest() -> pd.DataFrame:
    rows = []
    if not SEG_CQ500_ROOT.exists():
        return pd.DataFrame()
    for case_dir in sorted(p for p in SEG_CQ500_ROOT.iterdir() if p.is_dir()):
        img_path = case_dir / "CT.nii"
        mask_path = case_dir / "ICH_mask.nii.gz"
        if not img_path.exists() or not mask_path.exists():
            continue
        header = nifti_header_row(img_path)
        stats = mask_stats(mask_path, header["spacing"])
        rows.append(
            {
                "dataset": "Seg-CQ500",
                "role": "source",
                "task": "ich_segmentation",
                "patient_id": case_dir.name,
                "scan_id": case_dir.name,
                "img_path": str(img_path),
                "mask_path": str(mask_path),
                "source_license": "CC BY 4.0",
                "has_phe": "",
                "has_ich": bool(stats.get("mask_voxels", 0) > 0),
                **header,
                **stats,
            }
        )
    return pd.DataFrame(rows)


def build_instance_manifest() -> pd.DataFrame:
    rows = []
    data_dir = INSTANCE_ROOT / "data"
    label_dir = INSTANCE_ROOT / "label"
    if not data_dir.exists() or not label_dir.exists():
        return pd.DataFrame()
    images = {strip_nii_suffix(p): p for p in data_dir.iterdir() if p.is_file() and is_nifti(p)}
    masks = {strip_nii_suffix(p): p for p in label_dir.iterdir() if p.is_file() and is_nifti(p)}
    for scan_id, img_path in sorted(images.items()):
        mask_path = masks.get(scan_id)
        if not mask_path:
            continue
        header = nifti_header_row(img_path)
        stats = mask_stats(mask_path, header["spacing"])
        rows.append(
            {
                "dataset": "INSTANCE2022",
                "role": "source_optional",
                "task": "ich_segmentation",
                "patient_id": f"instance_{scan_id}",
                "scan_id": scan_id,
                "img_path": str(img_path),
                "mask_path": str(mask_path),
                "source_license": "Challenge data; verify terms before redistribution",
                "has_phe": "",
                "has_ich": bool(stats.get("mask_voxels", 0) > 0),
                **header,
                **stats,
            }
        )
    return pd.DataFrame(rows)


def build_physionet_manifest(root: Path = PHYSIONET_ROOT) -> pd.DataFrame:
    rows = []
    if not root.exists():
        return pd.DataFrame()
    image_candidates = []
    mask_candidates = []
    for p in root.rglob("*"):
        if not p.is_file() or not is_nifti(p):
            continue
        lower = str(p).lower()
        if "mask" in lower or "label" in lower or "seg" in lower:
            mask_candidates.append(p)
        elif "ct" in lower or "scan" in lower or "image" in lower:
            image_candidates.append(p)
    masks = {strip_nii_suffix(p): p for p in mask_candidates}
    for img_path in sorted(image_candidates):
        scan_id = strip_nii_suffix(img_path)
        mask_path = masks.get(scan_id)
        header = nifti_header_row(img_path)
        stats = mask_stats(mask_path, header["spacing"]) if mask_path else {}
        rows.append(
            {
                "dataset": "PhysioNet_CT_ICH",
                "role": "source_restricted",
                "task": "ich_segmentation",
                "patient_id": f"physionet_{scan_id}",
                "scan_id": scan_id,
                "img_path": str(img_path),
                "mask_path": str(mask_path) if mask_path else "",
                "source_license": "Restricted access + DUA; do not redistribute",
                "has_phe": "",
                "has_ich": bool(stats.get("mask_voxels", 0) > 0),
                **header,
                **stats,
            }
        )
    return pd.DataFrame(rows)


manifest_parts = [
    build_phe_manifest(),
    build_seg_cq500_manifest(),
    build_instance_manifest(),
    build_physionet_manifest(),
]
master_df = pd.concat([df for df in manifest_parts if len(df)], ignore_index=True)
schema_cols = [
    "dataset", "role", "task", "patient_id", "scan_id", "img_path", "mask_path", "source_license",
    "spacing_x", "spacing_y", "spacing_z", "n_slices", "has_ich", "has_phe",
    "mask_voxels", "mask_volume_ml", "positive_slices", "shape", "spacing",
]
master_df = master_df[[c for c in schema_cols if c in master_df.columns]]
master_path = MANIFEST_DIR / "subjects_master.csv"
master_df.to_csv(master_path, index=False, encoding="utf-8")

display(master_df.groupby(["dataset", "role", "task"]).agg(
    cases=("scan_id", "count"),
    slices=("n_slices", "sum"),
    median_spacing_z=("spacing_z", "median"),
    median_mask_ml=("mask_volume_ml", "median"),
).reset_index())
print("Saved:", master_path)


## 5. Patient-level folds và low-label subsets

Không chia theo slice. PHE-SICH được chia outer 5-fold ở mức patient/scan. Các subset 10/25/50/100% lấy từ train set theo strata thể tích PHE để tránh bias ca lớn hoặc ca dễ.


In [ ]:
def assign_strata(values: Sequence[float], n_bins: int = 4) -> np.ndarray:
    values = pd.Series(values).fillna(0.0).astype(float)
    if values.nunique() < 2:
        return np.zeros(len(values), dtype=int)
    try:
        return pd.qcut(values.rank(method="first"), q=min(n_bins, len(values)), labels=False).astype(int).to_numpy()
    except Exception:
        return pd.cut(values, bins=min(n_bins, len(values)), labels=False, include_lowest=True).fillna(0).astype(int).to_numpy()


def make_patient_folds(df: pd.DataFrame, n_folds: int = CFG.n_folds, seed: int = CFG.seed) -> pd.DataFrame:
    df = df.copy().reset_index(drop=True)
    strata = assign_strata(df.get("mask_volume_ml", pd.Series(np.zeros(len(df)))), n_bins=4)
    fold = np.full(len(df), -1, dtype=int)
    rng = np.random.default_rng(seed)
    for s in sorted(np.unique(strata)):
        idx = np.where(strata == s)[0]
        rng.shuffle(idx)
        for j, row_idx in enumerate(idx):
            fold[row_idx] = j % n_folds
    df["fold"] = fold
    df["volume_stratum"] = strata
    return df


def split_outer_fold(target_df: pd.DataFrame, fold_id: int = CFG.outer_fold, val_fraction: float = CFG.val_fraction, seed: int = CFG.seed):
    folded = target_df.copy()
    test_df = folded[folded["fold"] == fold_id].copy()
    trainval_df = folded[folded["fold"] != fold_id].copy()
    rng = np.random.default_rng(seed + fold_id)
    val_indices = []
    for s, part in trainval_df.groupby("volume_stratum"):
        idx = part.index.to_numpy().copy()
        rng.shuffle(idx)
        n_val = max(1, int(round(len(idx) * val_fraction))) if len(idx) > 4 else max(0, int(round(len(idx) * val_fraction)))
        val_indices.extend(idx[:n_val].tolist())
    val_df = trainval_df.loc[val_indices].copy()
    train_df = trainval_df.drop(index=val_indices).copy()
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


def make_low_label_subsets(train_df: pd.DataFrame, fractions=CFG.label_fractions, seed: int = CFG.seed) -> Dict[float, pd.DataFrame]:
    subsets = {}
    for frac in fractions:
        if frac >= 0.999:
            subsets[frac] = train_df.copy().reset_index(drop=True)
            continue
        rng = np.random.default_rng(seed + int(frac * 1000))
        chosen = []
        total_target = max(1, int(round(len(train_df) * frac)))
        for s, part in train_df.groupby("volume_stratum"):
            idx = part.index.to_numpy().copy()
            rng.shuffle(idx)
            n_take = max(1, int(round(len(idx) * frac)))
            chosen.extend(idx[:n_take].tolist())
        if len(chosen) > total_target:
            rng.shuffle(chosen)
            chosen = chosen[:total_target]
        subsets[frac] = train_df.loc[sorted(set(chosen))].copy().reset_index(drop=True)
    return subsets


phe_df = master_df[master_df["dataset"] == "PHE-SICH-CT-IDS"].copy()
phe_folds_df = make_patient_folds(phe_df)
fold_path = MANIFEST_DIR / "folds_phe_sich_v1.csv"
phe_folds_df.to_csv(fold_path, index=False, encoding="utf-8")

train_df, val_df, test_df = split_outer_fold(phe_folds_df, CFG.outer_fold)
low_label_subsets = make_low_label_subsets(train_df)

low_label_dir = MANIFEST_DIR / "low_label_subsets"
low_label_dir.mkdir(exist_ok=True)
for frac, subset in low_label_subsets.items():
    subset.to_csv(low_label_dir / f"phe_train_fold{CFG.outer_fold}_{int(frac * 100)}pct.csv", index=False, encoding="utf-8")

display(pd.DataFrame([
    {"split": "train", "cases": len(train_df), "median_phe_ml": train_df["mask_volume_ml"].median()},
    {"split": "val", "cases": len(val_df), "median_phe_ml": val_df["mask_volume_ml"].median()},
    {"split": "test", "cases": len(test_df), "median_phe_ml": test_df["mask_volume_ml"].median()},
]))
display(pd.DataFrame([
    {"label_fraction": frac, "cases": len(df), "median_phe_ml": df["mask_volume_ml"].median()}
    for frac, df in low_label_subsets.items()
]))
print("Saved folds:", fold_path)


## 5b. Locked 19c split for fair comparison

This overrides the generic CV split above with the exact validation/test scan IDs used by recent baselines. Keep this cell enabled so comparisons are fair, but do not let the baseline notebooks dictate the proposed method.


In [ ]:
# Locked 19c split for fair evaluation only.
USE_LOCKED_19C_SPLIT = True
VAL_SCAN_IDS = {'0013', '0014', '0051', '0060', '0061', '0078', '0113', '0116', '0160', '0167'}
TEST_SCAN_IDS = {'0002', '0029', '0033', '0080', '0084', '0095', '0096', '0115', '0119', '0130', '0138'}

if USE_LOCKED_19C_SPLIT:
    def _scan4(scan_id):
        s = str(scan_id).strip()
        return f'{int(s):04d}' if s.isdigit() else s

    split_df = phe_folds_df.copy().reset_index(drop=True)
    split_df['scan_id'] = split_df['scan_id'].map(_scan4)
    split_df['split'] = split_df['scan_id'].map(lambda s: 'test' if s in TEST_SCAN_IDS else ('val' if s in VAL_SCAN_IDS else 'train'))
    split_df['case_id'] = split_df['scan_id'].map(lambda s: f'phe_{int(s):04d}' if str(s).isdigit() else f'phe_{s}')
    train_df = split_df[split_df['split'] == 'train'].copy().reset_index(drop=True)
    val_df = split_df[split_df['split'] == 'val'].copy().reset_index(drop=True)
    test_df = split_df[split_df['split'] == 'test'].copy().reset_index(drop=True)
    low_label_subsets = make_low_label_subsets(train_df)

    locked_split_path = MANIFEST_DIR / '03_2_locked19c_phe_sich_split_manifest.csv'
    split_df.to_csv(locked_split_path, index=False, encoding='utf-8')
    low_label_dir.mkdir(exist_ok=True)
    for frac, subset in low_label_subsets.items():
        subset.to_csv(low_label_dir / f'03_2_locked19c_train_fold{CFG.outer_fold}_{int(frac * 100)}pct.csv', index=False, encoding='utf-8')

    display(split_df.groupby('split').agg(
        cases=('scan_id', 'count'),
        median_phe_ml=('mask_volume_ml', 'median'),
        min_phe_ml=('mask_volume_ml', 'min'),
        max_phe_ml=('mask_volume_ml', 'max'),
    ).reset_index())
    display(pd.DataFrame([
        {'label_fraction': int(frac * 100), 'cases': len(df), 'median_phe_ml': df['mask_volume_ml'].median() if len(df) else np.nan}
        for frac, df in low_label_subsets.items()
    ]))
    print('Locked split saved:', locked_split_path)
    print('19c validation IDs:', sorted(VAL_SCAN_IDS))
    print('19c test IDs      :', sorted(TEST_SCAN_IDS))
else:
    print('Using generic CV split from previous cell.')

## 6. Quick EDA và kiểm tra physical 2.5D

Cell này chọn lát có mask lớn nhất, hiển thị 3 window CT và kiểm tra các lát 2.5D được chọn theo offset mm.


In [ ]:
def largest_mask_slice(mask_volume: np.ndarray) -> int:
    if mask_volume.ndim != 3:
        return 0
    per_slice = mask_volume.reshape((-1, mask_volume.shape[2])).sum(axis=0)
    return int(np.argmax(per_slice))


def show_case_windows(row: pd.Series, image_size: int = 256):
    img, spacing, _ = load_nifti(Path(row["img_path"]))
    mask, _, _ = load_nifti(Path(row["mask_path"]))
    z = largest_mask_slice(mask)
    window_stack = make_2d_stack(img, z, image_size=image_size)
    mask_2d = resize_mask(np.asarray(mask[:, :, z]), image_size=image_size)
    indices = physical_neighbor_indices(z, img.shape[2], spacing[2])

    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    for c, (name, _, _) in enumerate(CFG.windows):
        axes[c].imshow(window_stack[c], cmap="gray")
        axes[c].imshow(np.ma.masked_where(mask_2d == 0, mask_2d), cmap="autumn", alpha=0.45)
        axes[c].set_title(name)
        axes[c].axis("off")
    axes[3].plot(CFG.physical_offsets_mm, indices, marker="o")
    axes[3].set_title(f"physical indices, z={z}, spacing_z={spacing[2]:.2f}mm")
    axes[3].set_xlabel("offset mm")
    axes[3].set_ylabel("slice index")
    axes[4].hist(mask.reshape((-1, mask.shape[2])).sum(axis=0), bins=20)
    axes[4].set_title("mask pixels/slice")
    plt.tight_layout()
    return fig


if len(phe_folds_df) and NIB_AVAILABLE and SCIPY_AVAILABLE:
    sample_row = phe_folds_df.sort_values("mask_volume_ml", ascending=False).iloc[0]
    fig = show_case_windows(sample_row, image_size=CFG.smoke_image_size)
    fig.savefig(FIG_DIR / "sample_phe_windows_physical_stack.png", dpi=160, bbox_inches="tight")
else:
    print("Skip visualization because PHE data, nibabel or scipy is unavailable.")


## 7b. Data Augmentation

Augmentation phù hợp với CT: flip ngang/dọc, rotation 90°, dịch/co cường độ nhẹ, và Gaussian noise. Tất cả biến đổi được áp dụng đồng bộ lên image, mask và prior. Chỉ dùng cho train set.

In [ ]:
class CTAugmentation:
    """CT-appropriate augmentation applied synchronously to image, mask and prior."""

    def __init__(self, prob: float = 0.5):
        self.prob = prob

    def __call__(self, image: np.ndarray, mask: np.ndarray, prior=None):
        """
        Args:
            image: (C, H, W) float32
            mask:  (H, W) float32
            prior: (H, W) float32 or None
        Returns:
            image, mask, prior (augmented copies)
        """
        # Random horizontal flip
        if random.random() < self.prob:
            image = np.flip(image, axis=2).copy()
            mask = np.flip(mask, axis=1).copy()
            if prior is not None:
                prior = np.flip(prior, axis=1).copy()

        # Random vertical flip
        if random.random() < self.prob:
            image = np.flip(image, axis=1).copy()
            mask = np.flip(mask, axis=0).copy()
            if prior is not None:
                prior = np.flip(prior, axis=0).copy()

        # Random 90-degree rotation
        k = random.randint(0, 3)
        if k > 0:
            image = np.rot90(image, k, axes=(1, 2)).copy()
            mask = np.rot90(mask, k, axes=(0, 1)).copy()
            if prior is not None:
                prior = np.rot90(prior, k, axes=(0, 1)).copy()

        # Random intensity shift/scale (conservative for medical imaging)
        if random.random() < self.prob:
            shift = np.float32(random.uniform(-0.03, 0.03))
            scale = np.float32(random.uniform(0.97, 1.03))
            image = np.clip(image * scale + shift, 0.0, 1.0).astype(np.float32)

        # Random Gaussian noise (low probability, subtle)
        if random.random() < self.prob * 0.4:
            noise = np.random.normal(0, 0.015, image.shape).astype(np.float32)
            image = np.clip(image + noise, 0.0, 1.0).astype(np.float32)

        return image, mask, prior


ct_augment = CTAugmentation(prob=CFG.aug_prob) if CFG.use_augmentation else None
print("Augmentation:", "ENABLED" if ct_augment else "disabled",
      f"(prob={CFG.aug_prob})" if ct_augment else "")

## 7. PyTorch Dataset cho 2D và 2.5D (with augmentation)

Dataset trả về:

- `image`: 3 channels cho 2D hoặc 15 channels cho 2.5D physical-aware.
- `mask`: mask trung tâm.
- `prior`: teacher prior nếu đã sinh, nếu chưa có thì dùng tensor 0 để smoke-test.
- `area_ml`: diện tích mask trung tâm quy đổi ml lát cắt bằng spacing gốc.

**NEW**: Hỗ trợ data augmentation qua tham số `augment`. Augmentation chỉ áp dụng cho train set.

In [ ]:
if TORCH_AVAILABLE:
    class SliceStackDataset(Dataset):
        def __init__(
            self,
            df: pd.DataFrame,
            mode: str = "2d",
            image_size: int = CFG.image_size,
            positive_only: bool = False,
            max_subjects: Optional[int] = None,
            prior_dir: Optional[Path] = None,
            augment=None,
        ):
            self.df = df.copy().reset_index(drop=True)
            if max_subjects is not None:
                self.df = self.df.iloc[:max_subjects].copy()
            self.mode = mode
            self.image_size = int(image_size)
            self.positive_only = positive_only
            self.prior_dir = Path(prior_dir) if prior_dir else None
            self.augment = augment
            self._volume_cache = {}
            self.index = self._build_index()

        def _load_pair(self, row_idx: int):
            if row_idx not in self._volume_cache:
                row = self.df.iloc[row_idx]
                image, spacing, _ = load_nifti(Path(row["img_path"]))
                mask, _, _ = load_nifti(Path(row["mask_path"]))
                self._volume_cache[row_idx] = (image.astype(np.float32), (mask > 0).astype(np.uint8), spacing)
            return self._volume_cache[row_idx]

        def _build_index(self):
            items = []
            for row_idx, row in self.df.iterrows():
                if not Path(row["mask_path"]).exists():
                    continue
                mask, spacing, _ = load_nifti(Path(row["mask_path"]))
                mask = np.asarray(mask) > 0
                n_slices = mask.shape[2]
                if self.positive_only:
                    z_indices = np.where(mask.reshape((-1, n_slices)).sum(axis=0) > 0)[0].tolist()
                    if not z_indices:
                        z_indices = list(range(n_slices))
                else:
                    z_indices = list(range(n_slices))
                items.extend((row_idx, int(z)) for z in z_indices)
            return items

        def __len__(self):
            return len(self.index)

        def _load_prior(self, row: pd.Series, z: int):
            if self.prior_dir is None:
                return np.zeros((self.image_size, self.image_size), dtype=np.float32)
            scan_id = str(row.get("scan_id", ""))
            case_id = str(row.get("case_id", ""))
            candidates = []
            if case_id:
                candidates.append(self.prior_dir / f"{case_id}.npz")
            if scan_id:
                candidates.append(self.prior_dir / f"{scan_id}.npz")
                if scan_id.isdigit():
                    candidates.append(self.prior_dir / f"phe_{int(scan_id):04d}.npz")
            prior_path = next((p for p in candidates if p.exists()), None)
            if prior_path is None:
                return np.zeros((self.image_size, self.image_size), dtype=np.float32)
            arr = np.load(prior_path)
            prior = arr["prior"]
            if prior.ndim == 3:
                prior = prior[:, :, z]
            return resize_2d(prior.astype(np.float32), self.image_size, order=1)

        def __getitem__(self, idx: int):
            row_idx, z = self.index[idx]
            row = self.df.iloc[row_idx]
            image, mask, spacing = self._load_pair(row_idx)
            if self.mode == "2d":
                x = make_2d_stack(image, z, image_size=self.image_size)
            elif self.mode in {"25d", "2.5d"}:
                x = make_25d_stack(image, z, spacing_z=spacing[2], image_size=self.image_size)
            else:
                raise ValueError(f"Unknown mode: {self.mode}")
            y = resize_mask(mask[:, :, z], image_size=self.image_size)
            prior = self._load_prior(row, z)

            # Apply data augmentation (train only)
            if self.augment is not None:
                x, y, prior = self.augment(x, y, prior)

            pixel_area_mm2 = float(spacing[0] * spacing[1])
            slice_area_ml = float(y.sum() * pixel_area_mm2 * spacing[2] / 1000.0)
            return {
                "image": torch.from_numpy(x).float(),
                "mask": torch.from_numpy(y[None]).float(),
                "prior": torch.from_numpy(prior[None]).float(),
                "area_ml": torch.tensor([slice_area_ml], dtype=torch.float32),
                "scan_id": str(row["scan_id"]),
                "slice_idx": int(z),
                "spacing": torch.tensor(spacing, dtype=torch.float32),
            }


    if CFG.run_smoke_test and len(train_df):
        smoke_ds = SliceStackDataset(
            train_df,
            mode="25d",
            image_size=CFG.smoke_image_size,
            positive_only=True,
            max_subjects=CFG.max_smoke_subjects,
            augment=ct_augment,
        )
        sample = smoke_ds[0]
        print({k: (tuple(v.shape) if hasattr(v, "shape") else v) for k, v in sample.items() if k in ["image", "mask", "prior", "area_ml"]})
        print("Smoke dataset slices:", len(smoke_ds))
else:
    print("Torch unavailable. Install torch to enable Dataset/DataLoader and training.")

## 8. Multi-head U-Net teacher/student (with SE attention + dropout)

Mô hình đủ minh bạch để tái lập:

- teacher học source ICH để dự đoán hemorrhage mask/prior.
- student học PHE mask, đồng thời học prior alignment, area regression và uncertainty log-variance.
- cùng backbone có thể chạy 2D (`in_channels=3`) hoặc 2.5D physical-aware (`in_channels=15`).

**NEW**: Squeeze-and-Excitation (SE) attention trong encoder blocks và Dropout sau bottleneck để giảm overfitting.

In [ ]:
if TORCH_AVAILABLE:
    class SEBlock(nn.Module):
        """Squeeze-and-Excitation channel attention."""
        def __init__(self, channels, reduction=16):
            super().__init__()
            mid = max(channels // reduction, 4)
            self.pool = nn.AdaptiveAvgPool2d(1)
            self.fc = nn.Sequential(
                nn.Linear(channels, mid, bias=False),
                nn.SiLU(inplace=True),
                nn.Linear(mid, channels, bias=False),
                nn.Sigmoid(),
            )

        def forward(self, x):
            b, c, _, _ = x.shape
            w = self.pool(x).view(b, c)
            w = self.fc(w).view(b, c, 1, 1)
            return x * w


    class DoubleConv(nn.Module):
        def __init__(self, in_ch, out_ch, use_se=True):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.SiLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.SiLU(inplace=True),
            )
            self.se = SEBlock(out_ch) if use_se else nn.Identity()

        def forward(self, x):
            return self.se(self.net(x))


    class TinyMultiTaskUNet(nn.Module):
        def __init__(self, in_channels=3, base=32, dropout=0.15):
            super().__init__()
            self.enc1 = DoubleConv(in_channels, base, use_se=False)
            self.enc2 = DoubleConv(base, base * 2, use_se=True)
            self.enc3 = DoubleConv(base * 2, base * 4, use_se=True)
            self.pool = nn.MaxPool2d(2)
            self.bottleneck = DoubleConv(base * 4, base * 8, use_se=True)
            self.drop = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()
            self.up3 = nn.ConvTranspose2d(base * 8, base * 4, kernel_size=2, stride=2)
            self.dec3 = DoubleConv(base * 8, base * 4, use_se=True)
            self.up2 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2)
            self.dec2 = DoubleConv(base * 4, base * 2, use_se=False)
            self.up1 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2)
            self.dec1 = DoubleConv(base * 2, base, use_se=False)
            self.seg_head = nn.Conv2d(base, 1, kernel_size=1)
            self.prior_head = nn.Conv2d(base, 1, kernel_size=1)
            self.log_var_head = nn.Conv2d(base, 1, kernel_size=1)
            self.area_head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(base, 1))

        def forward(self, x):
            e1 = self.enc1(x)
            e2 = self.enc2(self.pool(e1))
            e3 = self.enc3(self.pool(e2))
            b = self.drop(self.bottleneck(self.pool(e3)))
            d3 = self.up3(b)
            d3 = self.dec3(torch.cat([d3, e3], dim=1))
            d2 = self.up2(d3)
            d2 = self.dec2(torch.cat([d2, e2], dim=1))
            d1 = self.up1(d2)
            d1 = self.dec1(torch.cat([d1, e1], dim=1))
            return {
                "seg_logits": self.seg_head(d1),
                "prior_logits": self.prior_head(d1),
                "log_var": self.log_var_head(d1).clamp(-8.0, 8.0),
                "area_ml": F.softplus(self.area_head(d1)),
            }


    def build_model(mode: str = "2d", base: int = 32):
        in_channels = len(CFG.windows) if mode == "2d" else len(CFG.windows) * len(CFG.physical_offsets_mm)
        return TinyMultiTaskUNet(in_channels=in_channels, base=base)


    if CFG.run_smoke_test:
        model_25d = build_model("25d", base=16).to(DEVICE)
        n_params = sum(p.numel() for p in model_25d.parameters())
        x = torch.zeros(1, len(CFG.windows) * len(CFG.physical_offsets_mm), CFG.smoke_image_size, CFG.smoke_image_size, device=DEVICE)
        with torch.no_grad():
            out = model_25d(x)
        print({k: tuple(v.shape) for k, v in out.items()})
        print(f"Model params: {n_params:,} ({n_params/1e6:.2f}M)")
        del model_25d, x, out
else:
    print("Torch unavailable. Model definitions skipped.")

## 9. Loss: DiceFocal + boundary + prior + area + aleatoric uncertainty

Loss bám đúng công thức trong plan. Với teacher source ICH có thể dùng `seg_logits` làm hemorrhage mask. Với student PHE, `prior_logits` được align với soft prior từ teacher.


In [ ]:
if TORCH_AVAILABLE:
    def dice_loss_from_logits(logits, targets, eps=1e-6):
        probs = torch.sigmoid(logits)
        dims = tuple(range(1, probs.ndim))
        intersection = (probs * targets).sum(dims)
        union = probs.sum(dims) + targets.sum(dims)
        dice = (2.0 * intersection + eps) / (union + eps)
        return 1.0 - dice.mean()


    def focal_bce_from_logits(logits, targets, alpha=0.25, gamma=2.0):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p = torch.sigmoid(logits)
        pt = targets * p + (1 - targets) * (1 - p)
        alpha_t = targets * alpha + (1 - targets) * (1 - alpha)
        return (alpha_t * (1 - pt).pow(gamma) * bce).mean()


    def gradient_boundary_loss(logits, targets):
        probs = torch.sigmoid(logits)
        pred_dx = torch.abs(probs[:, :, :, 1:] - probs[:, :, :, :-1])
        pred_dy = torch.abs(probs[:, :, 1:, :] - probs[:, :, :-1, :])
        targ_dx = torch.abs(targets[:, :, :, 1:] - targets[:, :, :, :-1])
        targ_dy = torch.abs(targets[:, :, 1:, :] - targets[:, :, :-1, :])
        return F.l1_loss(pred_dx, targ_dx) + F.l1_loss(pred_dy, targ_dy)


    def aleatoric_bce_loss(seg_logits, targets, log_var):
        bce = F.binary_cross_entropy_with_logits(seg_logits, targets, reduction="none")
        precision = torch.exp(-log_var)
        return (precision * bce + log_var).mean()


    def multitask_loss(outputs, batch, weights=None):
        weights = weights or {"seg": 1.0, "bd": 0.1, "prior": 0.2, "area": 0.05, "unc": 0.1}
        y = batch["mask"].to(outputs["seg_logits"].device)
        prior = batch["prior"].to(outputs["seg_logits"].device)
        area = batch["area_ml"].to(outputs["seg_logits"].device)
        seg_loss = dice_loss_from_logits(outputs["seg_logits"], y) + focal_bce_from_logits(outputs["seg_logits"], y)
        bd_loss = gradient_boundary_loss(outputs["seg_logits"], y)
        prior_loss = F.mse_loss(torch.sigmoid(outputs["prior_logits"]), prior)
        area_loss = F.smooth_l1_loss(outputs["area_ml"], area)
        unc_loss = aleatoric_bce_loss(outputs["seg_logits"], y, outputs["log_var"])
        total = (
            weights["seg"] * seg_loss
            + weights["bd"] * bd_loss
            + weights["prior"] * prior_loss
            + weights["area"] * area_loss
            + weights["unc"] * unc_loss
        )
        return total, {
            "loss": float(total.detach().cpu()),
            "seg": float(seg_loss.detach().cpu()),
            "boundary": float(bd_loss.detach().cpu()),
            "prior": float(prior_loss.detach().cpu()),
            "area": float(area_loss.detach().cpu()),
            "unc": float(unc_loss.detach().cpu()),
        }
else:
    print("Torch unavailable. Loss functions skipped.")


## 10. Training loops và curriculum 3 giai đoạn (Optimized)

Curriculum theo plan:

1. **Pretrain teacher** trên Seg-CQ500 + INSTANCE cho binary ICH segmentation.
2. **Adapt teacher** trên PhysioNet nếu đã có DUA và dữ liệu local.
3. **Fine-tune student** trên PHE-SICH với 10/25/50/100% nhãn, dùng soft prior từ teacher.

**Optimizations** (NEW):
- Cosine annealing LR scheduler với linear warmup
- Early stopping theo val_dice với patience
- Best checkpoint auto-saving và auto-restore
- Gradient clipping để ổn định training
- Fixed AMP API (compatible PyTorch 1.x/2.x)
- Teacher validation split từ source ICH data

In [ ]:
if TORCH_AVAILABLE:
    class EarlyStopping:
        """Early stopping with patience tracking."""
        def __init__(self, patience: int = 10, mode: str = "max"):
            self.patience = patience
            self.mode = mode
            self.counter = 0
            self.best_score = None
            self.should_stop = False

        def __call__(self, score: float) -> bool:
            if self.best_score is None:
                self.best_score = score
                return False
            improved = (score > self.best_score) if self.mode == "max" else (score < self.best_score)
            if improved:
                self.best_score = score
                self.counter = 0
            else:
                self.counter += 1
                self.should_stop = self.counter >= self.patience
            return self.should_stop


    def get_cosine_schedule_with_warmup(optimizer, warmup_epochs: int, total_epochs: int, min_lr: float = 1e-6):
        """Cosine annealing with linear warmup."""
        base_lr = optimizer.param_groups[0]["lr"]
        def lr_lambda(epoch):
            if epoch < warmup_epochs:
                return max(min_lr / base_lr, (epoch + 1) / max(1, warmup_epochs))
            progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
            return max(min_lr / base_lr, 0.5 * (1 + math.cos(math.pi * progress)))
        return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


    def _make_amp_scaler():
        """Create GradScaler compatible with both PyTorch 1.x and 2.x."""
        try:
            return torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))
        except (TypeError, AttributeError):
            return torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))


    def _amp_autocast():
        """Create autocast context compatible with both PyTorch 1.x and 2.x."""
        try:
            return torch.amp.autocast("cuda", enabled=(DEVICE == "cuda"))
        except (TypeError, AttributeError):
            return torch.cuda.amp.autocast(enabled=(DEVICE == "cuda"))


    def train_one_epoch(model, loader, optimizer, device=DEVICE, scaler=None, grad_clip: float = 1.0):
        model.train()
        history = []
        for batch in loader:
            optimizer.zero_grad(set_to_none=True)
            x = batch["image"].to(device)
            if scaler is not None:
                with _amp_autocast():
                    outputs = model(x)
                    loss, parts = multitask_loss(outputs, batch)
                scaler.scale(loss).backward()
                if grad_clip > 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(x)
                loss, parts = multitask_loss(outputs, batch)
                loss.backward()
                if grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
            history.append(parts)
        return pd.DataFrame(history).mean(numeric_only=True).to_dict() if history else {}


    @torch.no_grad()
    def evaluate_slices(model, loader, threshold=0.5, device=DEVICE):
        model.eval()
        rows = []
        for batch in loader:
            x = batch["image"].to(device)
            y = batch["mask"].to(device)
            out = model(x)
            prob = torch.sigmoid(out["seg_logits"])
            pred = (prob >= threshold).float()
            dims = tuple(range(1, pred.ndim))
            inter = (pred * y).sum(dims)
            union = pred.sum(dims) + y.sum(dims)
            dice = ((2 * inter + 1e-6) / (union + 1e-6)).detach().cpu().numpy()
            rows.extend({"dice": float(v)} for v in dice)
        return pd.DataFrame(rows)


    def fit_model(model, train_loader, val_loader=None, epochs=1, lr=3e-4, ckpt_path=None,
                  patience=None, warmup_epochs=None, min_lr=None, grad_clip=None):
        """Train model with cosine LR, early stopping, best checkpoint and gradient clipping."""
        patience = patience if patience is not None else CFG.patience
        warmup_epochs = warmup_epochs if warmup_epochs is not None else CFG.warmup_epochs
        min_lr = min_lr if min_lr is not None else CFG.min_lr
        grad_clip = grad_clip if grad_clip is not None else CFG.grad_clip_norm

        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_epochs, epochs, min_lr)
        scaler = _make_amp_scaler()
        early_stop = EarlyStopping(patience=patience, mode="max")

        best_val_dice = -1.0
        best_state = None
        logs = []

        for epoch in range(1, epochs + 1):
            t0 = time.time()
            train_metrics = train_one_epoch(model, train_loader, optimizer, scaler=scaler, grad_clip=grad_clip)
            scheduler.step()

            row = {"epoch": epoch, "lr": round(float(optimizer.param_groups[0]["lr"]), 8),
                   **{f"train_{k}": v for k, v in train_metrics.items()}}

            if val_loader is not None:
                val_metrics = evaluate_slices(model, val_loader)
                val_dice = float(val_metrics["dice"].mean()) if len(val_metrics) else 0.0
                row["val_dice"] = round(val_dice, 5)

                if val_dice > best_val_dice:
                    best_val_dice = val_dice
                    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                    row["is_best"] = True

                if early_stop(val_dice):
                    row["time_s"] = round(time.time() - t0, 1)
                    logs.append(row)
                    print(f"  Early stopping at epoch {epoch}, best val_dice={best_val_dice:.4f}")
                    break

            row["time_s"] = round(time.time() - t0, 1)
            logs.append(row)

            if epoch == 1 or epoch % max(1, epochs // 10) == 0 or epoch == epochs:
                parts = " | ".join(
                    f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}"
                    for k, v in row.items() if k not in ("epoch", "is_best")
                )
                print(f"  [{epoch:3d}/{epochs}] {parts}")

        # Restore best checkpoint
        if best_state is not None:
            model.load_state_dict(best_state)
            print(f"  Restored best checkpoint (val_dice={best_val_dice:.4f})")

        if ckpt_path:
            ckpt_path.parent.mkdir(parents=True, exist_ok=True)
            torch.save({"model": model.state_dict(), "cfg": asdict(CFG), "logs": logs,
                         "best_val_dice": best_val_dice}, ckpt_path)
            print(f"  Saved: {ckpt_path}")

        return pd.DataFrame(logs)


    # ---- Teacher training section ----
    SOURCE_DATASET_POLICY = "Seg-CQ500 + INSTANCE if present + PhysioNet if present after DUA"
    source_df = master_df[master_df["task"] == "ich_segmentation"].copy()
    source_df = source_df[source_df["mask_path"].astype(str).str.len() > 0].copy()
    selected_teacher_ckpt = CHECKPOINT_DIR / f"03_2_teacher_source_{CFG.selected_mode}.pt"

    RUN_SELECTED_TEACHER_TRAINING = bool(CFG.run_heavy_training)

    print("Selected teacher policy:", SOURCE_DATASET_POLICY)
    print("ICH source cases available:", len(source_df))
    display(source_df.groupby("dataset").size().rename("cases").reset_index() if len(source_df) else pd.DataFrame())

    if RUN_SELECTED_TEACHER_TRAINING and len(source_df):
        # Split source data for teacher val
        source_shuffled = source_df.sample(frac=1.0, random_state=CFG.seed).reset_index(drop=True)
        n_teacher_val = max(1, int(len(source_shuffled) * CFG.teacher_val_fraction))
        source_val_df = source_shuffled.iloc[:n_teacher_val]
        source_train_df = source_shuffled.iloc[n_teacher_val:]
        print(f"Teacher split: train={len(source_train_df)}, val={len(source_val_df)}")

        teacher_train_ds = SliceStackDataset(source_train_df, mode=CFG.selected_mode,
                                              image_size=CFG.image_size, positive_only=False,
                                              augment=ct_augment if CFG.use_augmentation else None)
        teacher_val_ds = SliceStackDataset(source_val_df, mode=CFG.selected_mode,
                                            image_size=CFG.image_size, positive_only=False)
        teacher_train_loader = DataLoader(teacher_train_ds, batch_size=CFG.teacher_batch_size,
                                           shuffle=True, num_workers=NUM_WORKERS,
                                           pin_memory=(DEVICE == "cuda"))
        teacher_val_loader = DataLoader(teacher_val_ds, batch_size=CFG.teacher_batch_size,
                                         shuffle=False, num_workers=NUM_WORKERS,
                                         pin_memory=(DEVICE == "cuda"))
        teacher = build_model(CFG.selected_mode, base=32).to(DEVICE)
        teacher_logs = fit_model(
            teacher, teacher_train_loader, teacher_val_loader,
            epochs=CFG.teacher_epochs, lr=3e-4, ckpt_path=selected_teacher_ckpt,
        )
        teacher_logs.to_csv(LOG_DIR / f"03_2_teacher_source_{CFG.selected_mode}.csv", index=False)
    else:
        print("Selected teacher training skipped. Set CFG.run_heavy_training=True to run.")
        print("Expected teacher checkpoint:", selected_teacher_ckpt)

## 10b. Training curve visualization

Tự động vẽ training loss components, val_dice và learning rate qua epochs sau mỗi training run.

In [ ]:
def plot_training_curves(logs_df, title="Training Progress", save_path=None):
    """Plot training loss components, validation dice, and learning rate."""
    if logs_df is None or len(logs_df) == 0:
        print("No training logs to plot.")
        return None

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Plot 1: Total loss
    if "train_loss" in logs_df.columns:
        axes[0].plot(logs_df["epoch"], logs_df["train_loss"], "b-", linewidth=1.5, label="total loss")
        axes[0].set_title("Total Training Loss")
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel("Loss")
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

    # Plot 2: Loss components
    loss_cols = [c for c in logs_df.columns if c.startswith("train_") and c != "train_loss"]
    colors = plt.cm.Set2(np.linspace(0, 1, max(len(loss_cols), 1)))
    for col, color in zip(loss_cols, colors):
        axes[1].plot(logs_df["epoch"], logs_df[col], label=col.replace("train_", ""), color=color, linewidth=1)
    axes[1].set_title("Loss Components")
    axes[1].set_xlabel("Epoch")
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

    # Plot 3: Val dice + LR
    ax3 = axes[2]
    if "val_dice" in logs_df.columns:
        ax3.plot(logs_df["epoch"], logs_df["val_dice"], "g-o", markersize=3, linewidth=1.5, label="val_dice")
        if "is_best" in logs_df.columns:
            best_mask = logs_df["is_best"].fillna(False).astype(bool)
            if best_mask.any():
                best = logs_df[best_mask]
                ax3.scatter(best["epoch"], best["val_dice"], c="red", s=60, zorder=5, label="best", edgecolors="darkred")
        ax3.set_ylabel("Dice")
        ax3.legend(loc="lower right")
    ax3.set_title("Validation Dice & LR")
    ax3.set_xlabel("Epoch")
    ax3.grid(True, alpha=0.3)

    if "lr" in logs_df.columns:
        ax3r = ax3.twinx()
        ax3r.plot(logs_df["epoch"], logs_df["lr"], "r--", alpha=0.5, linewidth=1, label="LR")
        ax3r.set_ylabel("Learning Rate", color="red")
        ax3r.tick_params(axis="y", labelcolor="red")

    plt.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=160, bbox_inches="tight")
        print(f"Training curves saved: {save_path}")
    return fig


# Plot teacher curves if available
teacher_log_csv = LOG_DIR / f"03_2_teacher_source_{CFG.selected_mode}.csv"
if teacher_log_csv.exists():
    teacher_logs_loaded = pd.read_csv(teacher_log_csv)
    plot_training_curves(teacher_logs_loaded, title="Teacher ICH Training",
                         save_path=FIG_DIR / "teacher_training_curves.png")
else:
    print("Teacher training curves: not yet available (train teacher first).")

## 11. Sinh soft hemorrhage prior trên PHE-SICH

Prior được sinh từ teacher ICH trên ảnh PHE-SICH. Đây là khác biệt chính với hướng pseudo-label IPH từ edema mask: prior đến từ external ICH knowledge, không suy ngược từ nhãn PHE.


In [ ]:
if TORCH_AVAILABLE:
    @torch.no_grad()
    def predict_prior_volume(model, row: pd.Series, mode: str = "25d", image_size: int = CFG.image_size, device=DEVICE):
        model.eval()
        image, spacing, _ = load_nifti(Path(row["img_path"]))
        priors = []
        for z in range(image.shape[2]):
            x = make_25d_stack(image, z, spacing_z=spacing[2], image_size=image_size)
            xb = torch.from_numpy(x[None]).float().to(device)
            out = model(xb)
            prior = torch.sigmoid(out["seg_logits"])[0, 0].detach().cpu().numpy()
            prior = resize_2d(prior, image.shape[0], order=1)
            if prior.shape != image.shape[:2]:
                prior = ndimage.zoom(prior, (image.shape[0] / prior.shape[0], image.shape[1] / prior.shape[1]), order=1)
            priors.append(prior.astype(np.float16))
        return np.stack(priors, axis=2)


    def generate_selected_phe_priors_from_teacher(ckpt_path: Path, overwrite: bool = False):
        model = build_model(CFG.selected_mode, base=32).to(DEVICE)
        state = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(state["model"])
        rows = split_df.copy() if "split_df" in globals() else phe_folds_df.copy()
        for _, row in rows.iterrows():
            case_id = row.get("case_id", f"phe_{int(row['scan_id']):04d}" if str(row["scan_id"]).isdigit() else str(row["scan_id"]))
            out_path = PRIOR_DIR / f"{case_id}.npz"
            if out_path.exists() and not overwrite:
                continue
            prior = predict_prior_volume(model, row, mode=CFG.selected_mode, image_size=CFG.image_size)
            np.savez_compressed(out_path, prior=prior, source="external_ich_teacher")
            print("saved", out_path)
else:
    print("Torch unavailable. Prior generation helpers skipped.")


RUN_SELECTED_PRIOR_GENERATION = bool(CFG.run_heavy_training) and selected_teacher_ckpt.exists()
if TORCH_AVAILABLE and RUN_SELECTED_PRIOR_GENERATION:
    generate_selected_phe_priors_from_teacher(selected_teacher_ckpt, overwrite=False)
elif TORCH_AVAILABLE:
    print("Selected prior generation skipped until teacher checkpoint exists:", selected_teacher_ckpt)

## 12. Fine-tune the single selected PHE student


In [ ]:
if TORCH_AVAILABLE:
    def freeze_encoder_stem(model, freeze: bool = True):
        for module in [model.enc1, model.enc2]:
            for p in module.parameters():
                p.requires_grad = not freeze


    def init_encoder_from_teacher(student, teacher_ckpt: Path):
        if not teacher_ckpt.exists():
            print("No teacher checkpoint for encoder initialization:", teacher_ckpt)
            return student
        state = torch.load(teacher_ckpt, map_location=DEVICE)["model"]
        own = student.state_dict()
        keep_prefixes = ("enc", "bottleneck")
        compatible = {k: v for k, v in state.items() if k.startswith(keep_prefixes) and k in own and own[k].shape == v.shape}
        own.update(compatible)
        student.load_state_dict(own)
        print("Loaded teacher encoder weights:", len(compatible), "tensors")
        return student


    def run_selected_student_finetune(epochs: int = None):
        mode = CFG.selected_mode
        label_fraction = float(CFG.selected_label_fraction)
        train_subset = low_label_subsets[label_fraction]

        train_ds = SliceStackDataset(train_subset, mode=mode, image_size=CFG.image_size,
                                      positive_only=False, prior_dir=PRIOR_DIR,
                                      augment=ct_augment if CFG.use_augmentation else None)
        val_ds = SliceStackDataset(val_df, mode=mode, image_size=CFG.image_size,
                                    positive_only=False, prior_dir=PRIOR_DIR)
        train_loader = DataLoader(train_ds, batch_size=CFG.student_batch_size, shuffle=True,
                                   num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
        val_loader = DataLoader(val_ds, batch_size=CFG.student_batch_size, shuffle=False,
                                 num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))

        model = build_model(mode, base=32).to(DEVICE)
        model = init_encoder_from_teacher(model, selected_teacher_ckpt)
        if label_fraction in (0.10, 0.25):
            freeze_encoder_stem(model, True)

        ckpt = CHECKPOINT_DIR / f"03_2_student_phe_{mode}_fold{CFG.outer_fold}_{int(label_fraction * 100)}pct.pt"
        logs = fit_model(model, train_loader, val_loader,
                         epochs=epochs or CFG.student_epochs, lr=1e-4, ckpt_path=ckpt)
        logs.to_csv(LOG_DIR / f"03_2_student_phe_{mode}_fold{CFG.outer_fold}_{int(label_fraction * 100)}pct.csv", index=False)
        return model, logs, ckpt
else:
    print("Torch unavailable. Student fine-tuning helpers skipped.")


RUN_SELECTED_STUDENT_TRAINING = bool(CFG.run_heavy_training)
selected_student_ckpt = CHECKPOINT_DIR / f"03_2_student_phe_{CFG.selected_mode}_fold{CFG.outer_fold}_{int(CFG.selected_label_fraction * 100)}pct.pt"

if TORCH_AVAILABLE and RUN_SELECTED_STUDENT_TRAINING:
    selected_student_model, selected_student_logs, selected_student_ckpt = run_selected_student_finetune()
    plot_training_curves(selected_student_logs, title="Student PHE Training",
                         save_path=FIG_DIR / "student_training_curves.png")
elif TORCH_AVAILABLE:
    print("Selected student training skipped. Set CFG.run_heavy_training=True to run.")
    print("Expected student checkpoint:", selected_student_ckpt)
    # Plot existing student curves if available
    student_log_csv = LOG_DIR / f"03_2_student_phe_{CFG.selected_mode}_fold{CFG.outer_fold}_{int(CFG.selected_label_fraction * 100)}pct.csv"
    if student_log_csv.exists():
        plot_training_curves(pd.read_csv(student_log_csv), title="Student PHE Training (loaded)",
                             save_path=FIG_DIR / "student_training_curves.png")

## 13. Đánh giá volume-level, calibration và uncertainty

Mọi metric chính được tính ở mức subject/volume, không chỉ slice. Các hàm dưới đây hỗ trợ Dice, IoU, HD95, ASSD, NSD, volume MAE/RMSE, ECE, Brier và bootstrap CI.


In [ ]:
def binary_dice(pred: np.ndarray, target: np.ndarray, eps: float = 1e-6) -> float:
    pred = pred.astype(bool)
    target = target.astype(bool)
    return float((2 * np.logical_and(pred, target).sum() + eps) / (pred.sum() + target.sum() + eps))


def binary_iou(pred: np.ndarray, target: np.ndarray, eps: float = 1e-6) -> float:
    pred = pred.astype(bool)
    target = target.astype(bool)
    inter = np.logical_and(pred, target).sum()
    union = np.logical_or(pred, target).sum()
    return float((inter + eps) / (union + eps))


def volume_ml(mask: np.ndarray, spacing: Tuple[float, float, float]) -> float:
    return float((mask > 0).sum() * np.prod(spacing) / 1000.0)


def surface_distances(pred: np.ndarray, target: np.ndarray, spacing: Tuple[float, float, float]):
    if not SCIPY_AVAILABLE:
        return np.array([]), np.array([])
    pred = pred.astype(bool)
    target = target.astype(bool)
    if pred.sum() == 0 or target.sum() == 0:
        return np.array([np.inf]), np.array([np.inf])
    structure = ndimage.generate_binary_structure(3, 1)
    pred_surface = np.logical_xor(pred, ndimage.binary_erosion(pred, structure=structure, border_value=0))
    target_surface = np.logical_xor(target, ndimage.binary_erosion(target, structure=structure, border_value=0))
    dt_pred = ndimage.distance_transform_edt(~pred_surface, sampling=spacing)
    dt_target = ndimage.distance_transform_edt(~target_surface, sampling=spacing)
    return dt_target[pred_surface], dt_pred[target_surface]


def hd95_assd_nsd(pred: np.ndarray, target: np.ndarray, spacing: Tuple[float, float, float], nsd_tolerance_mm: float = 2.0):
    d1, d2 = surface_distances(pred, target, spacing)
    d = np.concatenate([d1, d2])
    finite = d[np.isfinite(d)]
    if len(finite) == 0:
        return {"hd95": np.inf, "assd": np.inf, "nsd": 0.0}
    return {
        "hd95": float(np.percentile(finite, 95)),
        "assd": float(finite.mean()),
        "nsd": float((finite <= nsd_tolerance_mm).mean()),
    }


def expected_calibration_error(probs: np.ndarray, targets: np.ndarray, n_bins: int = 15) -> float:
    probs = probs.reshape(-1).astype(np.float32)
    targets = targets.reshape(-1).astype(np.float32)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (probs >= lo) & (probs < hi if hi < 1.0 else probs <= hi)
        if not mask.any():
            continue
        conf = probs[mask].mean()
        acc = targets[mask].mean()
        ece += mask.mean() * abs(conf - acc)
    return float(ece)


def brier_score(probs: np.ndarray, targets: np.ndarray) -> float:
    return float(np.mean((probs.astype(np.float32) - targets.astype(np.float32)) ** 2))


def bootstrap_ci(values: Sequence[float], n_boot: int = 2000, alpha: float = 0.05, seed: int = CFG.seed):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return {"mean": np.nan, "lo": np.nan, "hi": np.nan}
    rng = np.random.default_rng(seed)
    boots = [rng.choice(values, size=len(values), replace=True).mean() for _ in range(n_boot)]
    return {
        "mean": float(values.mean()),
        "lo": float(np.percentile(boots, 100 * alpha / 2)),
        "hi": float(np.percentile(boots, 100 * (1 - alpha / 2))),
    }


def evaluate_probability_volume(prob: np.ndarray, target: np.ndarray, spacing: Tuple[float, float, float], threshold: float = 0.5) -> Dict:
    pred = prob >= threshold
    target_bin = target > 0
    pred_ml = volume_ml(pred, spacing)
    target_ml = volume_ml(target_bin, spacing)
    surface = hd95_assd_nsd(pred, target_bin, spacing)
    return {
        "dice": binary_dice(pred, target_bin),
        "iou": binary_iou(pred, target_bin),
        "pred_volume_ml": pred_ml,
        "target_volume_ml": target_ml,
        "volume_abs_error_ml": abs(pred_ml - target_ml),
        "ece": expected_calibration_error(prob, target_bin),
        "brier": brier_score(prob, target_bin),
        **surface,
    }


In [ ]:
if TORCH_AVAILABLE:
    @torch.no_grad()
    def predict_phe_probability_volume(model, row: pd.Series, image_size: int = CFG.image_size, device=DEVICE):
        model.eval()
        image, spacing, _ = load_nifti(Path(row["img_path"]))
        probs = []
        for z in range(image.shape[2]):
            x = make_25d_stack(image, z, spacing_z=spacing[2], image_size=image_size)
            xb = torch.from_numpy(x[None]).float().to(device)
            out = model(xb)
            prob = torch.sigmoid(out["seg_logits"])[0, 0].detach().cpu().numpy()
            prob = resize_2d(prob, image.shape[0], order=1)
            if prob.shape != image.shape[:2]:
                prob = ndimage.zoom(prob, (image.shape[0] / prob.shape[0], image.shape[1] / prob.shape[1]), order=1)
            probs.append(prob.astype(np.float32))
        return np.stack(probs, axis=2), spacing


    def save_prediction_mask_like_reference(mask_xyz: np.ndarray, reference_path: Path, out_path: Path):
        ref = nib.load(str(reference_path))
        mask_img = nib.Nifti1Image(mask_xyz.astype(np.uint8), ref.affine, ref.header)
        mask_img.set_data_dtype(np.uint8)
        nib.save(mask_img, str(out_path))


    def evaluate_selected_checkpoint_volume_level(ckpt_path: Path, threshold: float = 0.5):
        model = build_model(CFG.selected_mode, base=32).to(DEVICE)
        state = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(state["model"])
        model.eval()
        rows = []
        prob_dir = PRED_DIR / "03_2_selected_probabilities"
        mask_dir = PRED_DIR / "03_2_selected_masks"
        prob_dir.mkdir(parents=True, exist_ok=True)
        mask_dir.mkdir(parents=True, exist_ok=True)
        for _, row in test_df.iterrows():
            prob, spacing = predict_phe_probability_volume(model, row)
            target, _, _ = load_nifti(Path(row["mask_path"]))
            metrics = evaluate_probability_volume(prob, target, spacing, threshold=threshold)
            case_id = row.get("case_id", f"phe_{int(row['scan_id']):04d}" if str(row["scan_id"]).isdigit() else str(row["scan_id"]))
            pred_mask = (prob >= threshold).astype(np.uint8)
            np.savez_compressed(prob_dir / f"{case_id}.npz", prob=prob.astype(np.float16), threshold=np.float32(threshold))
            save_prediction_mask_like_reference(pred_mask, Path(row["img_path"]), mask_dir / f"{case_id}.nii.gz")
            rows.append({"case_id": case_id, "scan_id": row["scan_id"], "pred_mask_path": str(mask_dir / f"{case_id}.nii.gz"), **metrics})
        return pd.DataFrame(rows)
else:
    print("Torch unavailable. Checkpoint evaluation helpers skipped.")


RUN_SELECTED_VOLUME_EVAL = bool(TORCH_AVAILABLE and selected_student_ckpt.exists())
if RUN_SELECTED_VOLUME_EVAL:
    selected_volume_eval_df = evaluate_selected_checkpoint_volume_level(selected_student_ckpt, threshold=0.5)
    selected_volume_eval_df.to_csv(TABLE_DIR / "03_2_selected_student_volume_metrics.csv", index=False)
    print("Selected mask predictions:", PRED_DIR / "03_2_selected_masks")
    print("Selected probability maps:", PRED_DIR / "03_2_selected_probabilities")
    display(selected_volume_eval_df)
    display(pd.DataFrame({m: bootstrap_ci(selected_volume_eval_df[m]) for m in ["dice", "hd95", "nsd", "volume_abs_error_ml"]}).T)
elif TORCH_AVAILABLE:
    print("Selected volume evaluation skipped until checkpoint exists:", selected_student_ckpt)

## 14. One-switch full selected pipeline (Auto-connected)

This cell runs the chosen path end-to-end when `CFG.run_heavy_training=True`:
1. Teacher training (with val split + early stopping)
2. Prior generation on PHE-SICH
3. Student fine-tuning (with augmentation + early stopping)
4. Volume-level evaluation + prediction saving
5. Training curve plots
6. Summary report

**NEW**: Pipeline tự động kết nối theo `CFG.run_heavy_training` thay vì hardcoded `False`. Bao gồm training curves và summary report.

In [ ]:
RUN_FULL_SELECTED_PIPELINE = bool(CFG.run_heavy_training)

if RUN_FULL_SELECTED_PIPELINE:
    if not TORCH_AVAILABLE:
        raise RuntimeError("PyTorch is required for the full selected pipeline.")

    print("=" * 60)
    print("FULL PIPELINE: 2.5D Physical ICH-Prior Transfer")
    print(f"  Mode: {CFG.selected_mode} | Label fraction: {CFG.selected_label_fraction}")
    print(f"  Teacher epochs: {CFG.teacher_epochs} | Student epochs: {CFG.student_epochs}")
    print(f"  Patience: {CFG.patience} | Warmup: {CFG.warmup_epochs}")
    print("=" * 60)
    pipeline_t0 = time.time()

    # Stage 1: Teacher ICH Training
    if not selected_teacher_ckpt.exists():
        if len(source_df) == 0:
            raise RuntimeError("No ICH source cases found for teacher training.")
        print()
        print("--- Stage 1/4: Teacher ICH Training ---")
        source_shuffled = source_df.sample(frac=1.0, random_state=CFG.seed).reset_index(drop=True)
        n_tv = max(1, int(len(source_shuffled) * CFG.teacher_val_fraction))
        teacher_train_ds = SliceStackDataset(source_shuffled.iloc[n_tv:], mode=CFG.selected_mode,
                                              image_size=CFG.image_size, positive_only=False,
                                              augment=ct_augment if CFG.use_augmentation else None)
        teacher_val_ds = SliceStackDataset(source_shuffled.iloc[:n_tv], mode=CFG.selected_mode,
                                            image_size=CFG.image_size, positive_only=False)
        teacher_train_loader = DataLoader(teacher_train_ds, batch_size=CFG.teacher_batch_size,
                                           shuffle=True, num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
        teacher_val_loader = DataLoader(teacher_val_ds, batch_size=CFG.teacher_batch_size,
                                         shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
        teacher = build_model(CFG.selected_mode, base=32).to(DEVICE)
        teacher_logs = fit_model(teacher, teacher_train_loader, teacher_val_loader,
                                  epochs=CFG.teacher_epochs, lr=3e-4, ckpt_path=selected_teacher_ckpt)
        teacher_logs.to_csv(LOG_DIR / f"03_2_teacher_source_{CFG.selected_mode}.csv", index=False)
        plot_training_curves(teacher_logs, title="Teacher ICH Training",
                              save_path=FIG_DIR / "teacher_training_curves.png")
    else:
        print()
        print("--- Stage 1/4: Using existing teacher checkpoint ---")
        print(" ", selected_teacher_ckpt)

    # Stage 2: Generate hemorrhage priors
    print()
    print("--- Stage 2/4: Generate Hemorrhage Priors ---")
    generate_selected_phe_priors_from_teacher(selected_teacher_ckpt, overwrite=False)

    # Stage 3: Student PHE Fine-tuning
    if not selected_student_ckpt.exists():
        print()
        print("--- Stage 3/4: Student PHE Fine-tuning ---")
        selected_student_model, selected_student_logs, selected_student_ckpt = run_selected_student_finetune()
        plot_training_curves(selected_student_logs, title="Student PHE Training",
                              save_path=FIG_DIR / "student_training_curves.png")
    else:
        print()
        print("--- Stage 3/4: Using existing student checkpoint ---")
        print(" ", selected_student_ckpt)

    # Stage 4: Volume-level evaluation
    print()
    print("--- Stage 4/4: Volume-Level Evaluation ---")
    selected_volume_eval_df = evaluate_selected_checkpoint_volume_level(selected_student_ckpt, threshold=0.5)
    selected_volume_eval_df.to_csv(TABLE_DIR / "03_2_selected_student_volume_metrics.csv", index=False)

    elapsed = time.time() - pipeline_t0
    print()
    print("=" * 60)
    print(f"PIPELINE COMPLETED in {elapsed/60:.1f} minutes")
    print("=" * 60)
    display(selected_volume_eval_df)
    ci_metrics = ["dice", "hd95", "nsd", "volume_abs_error_ml"]
    available_ci = [m for m in ci_metrics if m in selected_volume_eval_df.columns]
    if available_ci:
        display(pd.DataFrame({m: bootstrap_ci(selected_volume_eval_df[m]) for m in available_ci}).T)
else:
    print("Full pipeline ready. To run:")
    print("  1. Set CFG.run_heavy_training = True in the config cell")
    print("  2. Re-run all cells (or set RUN_FULL_SELECTED_PIPELINE = True here)")
    print()
    print("Or run sections 10-14 step-by-step for more control.")

## 14b. Post-training analysis

Phân tích chi tiết sau khi chạy training pipeline: calibration plot, volume correlation scatter plot, và qualitative visualization top/bottom Dice cases.

In [ ]:
def plot_calibration_curve(probs_flat, targets_flat, n_bins=15, save_path=None):
    """Plot reliability diagram for probability calibration."""
    bins = np.linspace(0, 1, n_bins + 1)
    means, accs = [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (probs_flat >= lo) & (probs_flat < (hi if hi < 1.0 else hi + 1e-6))
        if mask.any():
            means.append(float(probs_flat[mask].mean()))
            accs.append(float(targets_flat[mask].mean()))

    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Perfect calibration")
    if means:
        ax.plot(means, accs, "bo-", linewidth=1.5, label="Model")
    ax.set_xlabel("Mean Predicted Probability")
    ax.set_ylabel("Fraction of Positives")
    ax.set_title("Calibration / Reliability Diagram")
    ax.legend()
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.grid(True, alpha=0.3)
    ax.set_aspect("equal")
    if save_path:
        fig.savefig(save_path, dpi=160, bbox_inches="tight")
    return fig


def plot_volume_scatter(eval_df, save_path=None):
    """Plot predicted vs ground truth volume."""
    if "target_volume_ml" not in eval_df.columns or "pred_volume_ml" not in eval_df.columns:
        print("Missing volume columns for scatter plot.")
        return None

    fig, ax = plt.subplots(1, 1, figsize=(7, 7))
    gt = eval_df["target_volume_ml"].astype(float)
    pred = eval_df["pred_volume_ml"].astype(float)
    ax.scatter(gt, pred, c="steelblue", s=60, alpha=0.7, edgecolors="navy", linewidths=0.5)

    lims = [0, max(gt.max(), pred.max()) * 1.1 + 0.1]
    ax.plot(lims, lims, "k--", alpha=0.5, label="y = x")
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_xlabel("Ground Truth Volume (mL)")
    ax.set_ylabel("Predicted Volume (mL)")
    ax.set_title("Volume Prediction Scatter")
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_aspect("equal")

    if len(gt) >= 2:
        r = float(np.corrcoef(gt, pred)[0, 1])
        mae = float(np.abs(gt - pred).mean())
        ax.text(0.05, 0.92, f"Pearson r = {r:.3f}\nMAE = {mae:.2f} mL",
                transform=ax.transAxes, fontsize=11, va="top",
                bbox=dict(boxstyle="round,pad=0.4", facecolor="lightyellow", alpha=0.8))

    if save_path:
        fig.savefig(save_path, dpi=160, bbox_inches="tight")
    return fig


def show_qualitative_results(eval_df, test_df_ref, n=3, save_path=None):
    """Show top/bottom Dice cases side by side: CT, ground truth overlay, prediction overlay."""
    if not NIB_AVAILABLE or "dice" not in eval_df.columns:
        print("Qualitative visualization requires nibabel and dice column.")
        return None

    eval_sorted = eval_df.dropna(subset=["dice"]).sort_values("dice", ascending=False).reset_index(drop=True)
    if len(eval_sorted) == 0:
        return None

    n = min(n, len(eval_sorted))
    top_cases = eval_sorted.head(n)
    bottom_cases = eval_sorted.tail(n)
    show_cases = pd.concat([top_cases, bottom_cases]).drop_duplicates(subset=["case_id"]).reset_index(drop=True)

    nrows = len(show_cases)
    fig, axes = plt.subplots(nrows, 3, figsize=(14, 4 * nrows))
    if nrows == 1:
        axes = axes[None, :]

    for i, (_, row) in enumerate(show_cases.iterrows()):
        case_id = row.get("case_id", "?")
        dice_val = row.get("dice", 0)
        pred_path = row.get("pred_mask_path", "")

        if not pred_path or not Path(pred_path).exists():
            for ax in axes[i]:
                ax.text(0.5, 0.5, f"{case_id}: no prediction", transform=ax.transAxes, ha="center")
                ax.axis("off")
            continue

        scan_id = row.get("scan_id", case_id)
        img_row = test_df_ref[test_df_ref["scan_id"].astype(str) == str(scan_id)]
        if len(img_row) == 0:
            for ax in axes[i]:
                ax.text(0.5, 0.5, f"{case_id}: not in test set", transform=ax.transAxes, ha="center")
                ax.axis("off")
            continue

        img_row = img_row.iloc[0]
        ct, spacing, _ = load_nifti(Path(img_row["img_path"]))
        gt, _, _ = load_nifti(Path(img_row["mask_path"]))
        pred_vol, _, _ = load_nifti(Path(pred_path))

        z = largest_mask_slice(gt) if gt.sum() > 0 else ct.shape[2] // 2

        ct_slice = window_ct(ct[:, :, z], -20, 100)
        gt_slice = gt[:, :, z] > 0
        pred_slice = pred_vol[:, :, z] > 0

        label = "TOP" if i < n else "BOTTOM"

        axes[i, 0].imshow(ct_slice, cmap="gray")
        axes[i, 0].set_title(f"{label} | {case_id} | Dice={dice_val:.3f}", fontsize=10)
        axes[i, 0].axis("off")

        axes[i, 1].imshow(ct_slice, cmap="gray")
        axes[i, 1].imshow(np.ma.masked_where(~gt_slice, gt_slice.astype(float)), cmap="Greens", alpha=0.5)
        axes[i, 1].set_title("Ground Truth", fontsize=10)
        axes[i, 1].axis("off")

        axes[i, 2].imshow(ct_slice, cmap="gray")
        axes[i, 2].imshow(np.ma.masked_where(~pred_slice, pred_slice.astype(float)), cmap="Reds", alpha=0.5)
        axes[i, 2].set_title("Prediction", fontsize=10)
        axes[i, 2].axis("off")

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=160, bbox_inches="tight")
        print(f"Qualitative results saved: {save_path}")
    return fig


# ---- Run post-analysis if results exist ----
POST_ANALYSIS_CSV = TABLE_DIR / "03_2_selected_student_volume_metrics.csv"
if POST_ANALYSIS_CSV.exists():
    post_df = pd.read_csv(POST_ANALYSIS_CSV)
    print(f"Post-analysis: {len(post_df)} test cases")

    plot_volume_scatter(post_df, save_path=FIG_DIR / "volume_scatter.png")
    show_qualitative_results(post_df, test_df, n=3, save_path=FIG_DIR / "qualitative_top_bottom.png")

    # Summary statistics table
    metric_cols = ["dice", "iou", "hd95", "assd", "nsd", "volume_abs_error_ml", "ece", "brier"]
    available = [c for c in metric_cols if c in post_df.columns]
    if available:
        summary = post_df[available].describe().T
        print()
        display(summary)
else:
    print(f"Post-analysis skipped. Run training pipeline first.")
    print(f"Expected: {POST_ANALYSIS_CSV}")

## 15. Reproducibility pack

Các artifact tối thiểu trước khi nộp bài:

- `subjects_master.csv`, `folds_phe_sich_v1.csv`, low-label subset CSV.
- Config đầy đủ, seed, package versions.
- Logs/checkpoint theo fold.
- Bảng metric patient-level kèm bootstrap CI.
- Data card/model card và license notes, đặc biệt với PhysioNet DUA.


In [ ]:
def package_versions() -> Dict[str, str]:
    versions = {
        "python": ".".join(map(str, os.sys.version_info[:3])),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    }
    if NIB_AVAILABLE:
        versions["nibabel"] = nib.__version__
    if SCIPY_AVAILABLE:
        import scipy
        versions["scipy"] = scipy.__version__
    if TORCH_AVAILABLE:
        versions["torch"] = torch.__version__
    return versions


run_metadata = {
    "config": {
        **asdict(CFG),
        "project_root": str(PROJECT_ROOT),
        "output_root": str(OUTPUT_ROOT),
        "windows": list(CFG.windows),
        "physical_offsets_mm": list(CFG.physical_offsets_mm),
    },
    "package_versions": package_versions(),
    "datasets": audit_df.to_dict(orient="records"),
    "manifest_path": str(master_path),
    "fold_path": str(fold_path),
    "research_constraints": [
        "No RSNA/BHSD in core track",
        "No pseudo-IPH labels inferred from PHE masks",
        "Patient-level splits only",
        "Report volume-level metrics with uncertainty/calibration",
    ],
}

with open(LOG_DIR / "run_metadata.json", "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, ensure_ascii=False, indent=2, default=str)

data_card = f"""
# Data Card - Q1 PHE/ICH Prior Transfer

Target dataset:
- PHE-SICH-CT-IDS SubdatasetA NIfTI, role: target PHE segmentation.

Source datasets:
- Seg-CQ500, role: source ICH segmentation.
- INSTANCE 2022 train labels, role: optional source extension, verify challenge terms.
- PhysioNet CT-ICH, role: optional restricted source after DUA.

Split policy:
- Patient/scan-level 5-fold CV.
- Low-label subsets are sampled from train subjects only.

Core exclusions:
- RSNA/BHSD are not part of the core paper track.
- No pseudo-IPH labels are inferred from PHE edema masks.
"""
(LOG_DIR / "DATA_CARD.md").write_text(data_card, encoding="utf-8")

model_card = """
# Model Card - 03_2 Selected 2.5D Physical Teacher Student PHE Segmentation

Architecture:
- Selected multi-head U-Net reference implementation for the chosen 2.5D physical path.
- Teacher: binary ICH segmentation on source datasets.
- Student: PHE segmentation with prior alignment, area regression and uncertainty log-variance.

Inputs:
- 2D: 3 CT windows.
- 2.5D: 5 physical offsets x 3 CT windows.

Outputs:
- PHE probability map.
- Prior alignment map.
- Slice area prediction.
- Uncertainty log-variance map.

Evaluation:
- Patient-level Dice, IoU, HD95, ASSD, NSD.
- Volume MAE/RMSE/correlation.
- Pixel-wise ECE, Brier score and calibration analysis.
"""
(LOG_DIR / "MODEL_CARD.md").write_text(model_card, encoding="utf-8")

print("Saved reproducibility pack to:", LOG_DIR)


## Next actions for the selected path

1. Run audit/manifest/locked split cells.
2. Put PhysioNet data under `PhysioNet_CT_ICH/` only if DUA access is ready.
3. Set `CFG.run_heavy_training=True` in the config cell.
4. Either run sections 10-14 step-by-step, or set `RUN_FULL_SELECTED_PIPELINE=True` in the one-switch runner.
5. Use the fair evaluator only after `predictions/03_2_selected_masks` exists.

Do not add a new baseline here unless it directly tests the selected method. Baseline notebooks can be evaluated outside this method notebook with the fair evaluator.


## Fair volume-level evaluator for optional external baselines

Use this instead of comparing raw notebook summaries. It creates one locked `labelsTs` folder, evaluates every prediction folder case-by-case, and reports patient-level Dice/IoU/HD95/ASSD/NSD plus volume errors with bootstrap CIs. This section is for fair reporting only; it is not the method design path.


In [ ]:
# Fair evaluator: same locked test labels and same patient-level metrics for any optional external baseline.
import shutil
try:
    import SimpleITK as sitk
except Exception:
    sitk = None
try:
    import nibabel as nib
except Exception:
    nib = None
if sitk is None and nib is None:
    raise ImportError('Fair NIfTI evaluation requires either SimpleITK or nibabel.')
from scipy import ndimage, stats as scipy_stats

FAIR_EVAL_DIR = OUTPUT_ROOT / 'fair_eval'
FAIR_LABELS_DIR = FAIR_EVAL_DIR / 'labelsTs_19c'
FAIR_TABLE_DIR = TABLE_DIR / 'fair_eval'
FAIR_LABELS_DIR.mkdir(parents=True, exist_ok=True)
FAIR_TABLE_DIR.mkdir(parents=True, exist_ok=True)


def read_mask(path):
    path = Path(path)
    if sitk is not None:
        img = sitk.ReadImage(str(path))
        arr = sitk.GetArrayFromImage(img) > 0  # z, y, x
        spacing_xyz = tuple(float(v) for v in img.GetSpacing())
        return img, arr, spacing_xyz
    img = nib.load(str(path))
    arr_xyz = np.asanyarray(img.dataobj) > 0
    if arr_xyz.ndim != 3:
        raise ValueError(f'Expected 3D NIfTI mask, got {arr_xyz.shape} for {path}')
    arr_zyx = np.transpose(arr_xyz, (2, 1, 0))
    spacing_xyz = tuple(float(v) for v in img.header.get_zooms()[:3])
    return img, arr_zyx, spacing_xyz


def write_mask_like(mask, reference_path, out_path):
    out_path = Path(out_path)
    reference_path = Path(reference_path)
    if sitk is not None:
        ref = sitk.ReadImage(str(reference_path))
        out = sitk.GetImageFromArray(mask.astype(np.uint8))
        if out.GetSize() != ref.GetSize():
            raise ValueError(f'Size mismatch for {out_path}: {out.GetSize()} vs {ref.GetSize()}')
        out.CopyInformation(ref)
        sitk.WriteImage(out, str(out_path))
        return
    # nibabel fallback: copy the original mask file into labelsTs. read_mask binarizes it later.
    shutil.copy2(reference_path, out_path)


def build_fair_labels(overwrite=True):
    if overwrite and FAIR_LABELS_DIR.exists():
        for p in FAIR_LABELS_DIR.glob('*.nii.gz'):
            p.unlink()
    rows = []
    for row in test_df.itertuples(index=False):
        case_id = str(getattr(row, 'case_id', f'phe_{int(row.scan_id):04d}'))
        _, mask, _ = read_mask(Path(row.mask_path))
        out_path = FAIR_LABELS_DIR / f'{case_id}.nii.gz'
        if sitk is not None:
            write_mask_like(mask, Path(row.img_path), out_path)
        else:
            shutil.copy2(Path(row.mask_path), out_path)
        rows.append({'case_id': case_id, 'scan_id': row.scan_id, 'label_path': str(out_path), 'source_mask_path': row.mask_path})
    df = pd.DataFrame(rows)
    df.to_csv(FAIR_EVAL_DIR / '03_2_fair_labelsTs_19c_manifest.csv', index=False)
    return df


def safe_div(a, b):
    return np.nan if b == 0 else float(a / b)


def overlap(pred, gt):
    pred = pred.astype(bool); gt = gt.astype(bool)
    tp = int(np.logical_and(pred, gt).sum())
    fp = int(np.logical_and(pred, ~gt).sum())
    fn = int(np.logical_and(~pred, gt).sum())
    tn = int(np.logical_and(~pred, ~gt).sum())
    pred_pos = tp + fp; gt_pos = tp + fn
    if pred_pos == 0 and gt_pos == 0:
        dice = iou = precision = recall = np.nan
    elif pred_pos == 0 or gt_pos == 0:
        dice = iou = 0.0
        precision = 0.0 if pred_pos else np.nan
        recall = 0.0 if gt_pos else np.nan
    else:
        dice = safe_div(2 * tp, 2 * tp + fp + fn)
        iou = safe_div(tp, tp + fp + fn)
        precision = safe_div(tp, pred_pos)
        recall = safe_div(tp, gt_pos)
    return {'dice': dice, 'iou': iou, 'precision': precision, 'recall': recall, 'specificity': safe_div(tn, tn + fp), 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn}


def diag_mm(shape_zyx, spacing_xyz):
    spacing_zyx = (float(spacing_xyz[2]), float(spacing_xyz[1]), float(spacing_xyz[0]))
    return float(np.sqrt(sum(((int(s) - 1) * sp) ** 2 for s, sp in zip(shape_zyx, spacing_zyx))))


def surface(mask):
    mask = mask.astype(bool)
    if not mask.any():
        return mask
    st = ndimage.generate_binary_structure(mask.ndim, 1)
    return np.logical_xor(mask, ndimage.binary_erosion(mask, structure=st, border_value=0))


def boundary_metrics(pred, gt, spacing_xyz, nsd_tol_mm=2.0):
    pred = pred.astype(bool); gt = gt.astype(bool)
    if not pred.any() and not gt.any():
        return {'hd95_mm': np.nan, 'assd_mm': np.nan, 'nsd_2mm': np.nan}
    if not pred.any() or not gt.any():
        penalty = diag_mm(pred.shape, spacing_xyz)
        return {'hd95_mm': penalty, 'assd_mm': penalty, 'nsd_2mm': 0.0}
    pred_s = surface(pred); gt_s = surface(gt)
    sampling = (float(spacing_xyz[2]), float(spacing_xyz[1]), float(spacing_xyz[0]))
    d_to_gt = ndimage.distance_transform_edt(~gt_s, sampling=sampling)[pred_s]
    d_to_pred = ndimage.distance_transform_edt(~pred_s, sampling=sampling)[gt_s]
    all_d = np.concatenate([d_to_gt, d_to_pred]).astype(float)
    close = int((d_to_gt <= nsd_tol_mm).sum() + (d_to_pred <= nsd_tol_mm).sum())
    denom = int(len(d_to_gt) + len(d_to_pred))
    return {'hd95_mm': float(np.percentile(all_d, 95)), 'assd_mm': float(all_d.mean()), 'nsd_2mm': safe_div(close, denom)}


def volume_ml(mask, spacing_xyz):
    return float(mask.astype(bool).sum() * np.prod(np.asarray(spacing_xyz, dtype=float)) / 1000.0)


def evaluate_case(case_id, pred_path, label_path):
    _, gt, spacing_xyz = read_mask(label_path)
    _, pred, _ = read_mask(pred_path)
    if gt.shape != pred.shape:
        raise ValueError(f'{case_id}: shape mismatch gt={gt.shape} pred={pred.shape}')
    gt_ml = volume_ml(gt, spacing_xyz)
    pred_ml = volume_ml(pred, spacing_xyz)
    rvd = np.nan if gt_ml == 0 and pred_ml == 0 else (np.inf if gt_ml == 0 else float((pred_ml - gt_ml) / gt_ml))
    return {
        'case_id': case_id,
        'gt_volume_ml': gt_ml,
        'pred_volume_ml': pred_ml,
        'volume_error_ml': float(pred_ml - gt_ml),
        'volume_abs_error_ml': float(abs(pred_ml - gt_ml)),
        'volume_sq_error_ml2': float((pred_ml - gt_ml) ** 2),
        'rvd_signed': rvd,
        'rvd_abs': abs(rvd) if np.isfinite(rvd) else rvd,
        'gt_positive': bool(gt.any()),
        'pred_positive': bool(pred.any()),
        **overlap(pred, gt),
        **boundary_metrics(pred, gt, spacing_xyz, nsd_tol_mm=2.0),
    }


def bootstrap_ci(values, reducer=np.nanmean, n_boot=2000, seed=42):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return {'mean': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'n': 0}
    rng = np.random.default_rng(seed)
    boots = [float(reducer(rng.choice(values, size=len(values), replace=True))) for _ in range(n_boot)]
    return {'mean': float(reducer(values)), 'ci_low': float(np.percentile(boots, 2.5)), 'ci_high': float(np.percentile(boots, 97.5)), 'n': int(len(values))}


def summarize_fair(per_case_df, source_name):
    rows = []
    for metric in ['dice', 'iou', 'precision', 'recall', 'hd95_mm', 'assd_mm', 'nsd_2mm', 'volume_abs_error_ml', 'rvd_abs']:
        if metric in per_case_df.columns:
            ci = bootstrap_ci(pd.to_numeric(per_case_df[metric], errors='coerce'))
            rows.append({'source': source_name, 'metric': metric, **ci})
    if 'volume_sq_error_ml2' in per_case_df.columns:
        ci = bootstrap_ci(pd.to_numeric(per_case_df['volume_sq_error_ml2'], errors='coerce'), reducer=lambda x: float(np.sqrt(np.nanmean(x))))
        rows.append({'source': source_name, 'metric': 'volume_rmse_ml', **ci})
    if {'gt_volume_ml', 'pred_volume_ml'}.issubset(per_case_df.columns):
        gt = pd.to_numeric(per_case_df['gt_volume_ml'], errors='coerce')
        pred = pd.to_numeric(per_case_df['pred_volume_ml'], errors='coerce')
        ok = gt.notna() & pred.notna()
        if int(ok.sum()) >= 2:
            rows.append({'source': source_name, 'metric': 'volume_pearson_r', 'mean': float(np.corrcoef(gt[ok], pred[ok])[0, 1]), 'ci_low': np.nan, 'ci_high': np.nan, 'n': int(ok.sum())})
            rows.append({'source': source_name, 'metric': 'volume_spearman_rho', 'mean': float(scipy_stats.spearmanr(gt[ok], pred[ok]).correlation) if 'scipy_stats' in globals() else np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'n': int(ok.sum())})
    return pd.DataFrame(rows)


def evaluate_prediction_folder(pred_dir, source_name, labels_dir=FAIR_LABELS_DIR, case_ids=None):
    pred_dir = Path(pred_dir)
    labels_dir = Path(labels_dir)
    case_ids = list(case_ids) if case_ids is not None else [p.name[:-7] for p in sorted(labels_dir.glob('*.nii.gz'))]
    rows, missing = [], []
    for case_id in case_ids:
        label_path = labels_dir / f'{case_id}.nii.gz'
        pred_path = pred_dir / f'{case_id}.nii.gz'
        if not label_path.exists() or not pred_path.exists():
            missing.append({'source': source_name, 'case_id': case_id, 'missing_label': not label_path.exists(), 'missing_pred': not pred_path.exists()})
            continue
        try:
            rows.append({'source': source_name, **evaluate_case(case_id, pred_path, label_path)})
        except Exception as exc:
            missing.append({'source': source_name, 'case_id': case_id, 'error': str(exc)})
    per_case = pd.DataFrame(rows)
    summary = summarize_fair(per_case, source_name) if len(per_case) else pd.DataFrame()
    missing_df = pd.DataFrame(missing)
    per_case.to_csv(FAIR_TABLE_DIR / f'{source_name}_per_case.csv', index=False)
    summary.to_csv(FAIR_TABLE_DIR / f'{source_name}_bootstrap_summary.csv', index=False)
    if len(missing_df):
        missing_df.to_csv(FAIR_TABLE_DIR / f'{source_name}_missing.csv', index=False)
    print(source_name, 'valid cases:', len(per_case), 'missing/errors:', len(missing_df))
    return per_case, summary, missing_df


fair_labels_df = build_fair_labels(overwrite=True)
print('Fair labels folder:', FAIR_LABELS_DIR)
display(fair_labels_df)

PREDICTION_SOURCES = {
    '03_2_selected_student': PRED_DIR / '03_2_selected_masks',
    # Optional: add external baseline prediction folders manually if you want fair reporting.
}

RUN_FAIR_EVAL_NOW = False
if RUN_FAIR_EVAL_NOW:
    all_cases, all_summaries = [], []
    ids = fair_labels_df['case_id'].astype(str).tolist()
    for source, folder in PREDICTION_SOURCES.items():
        if not Path(folder).exists():
            print('Skip missing folder:', source, folder)
            continue
        per_case, summary, missing = evaluate_prediction_folder(folder, source, case_ids=ids)
        all_cases.append(per_case)
        all_summaries.append(summary)
    if all_summaries:
        combined = pd.concat(all_summaries, ignore_index=True)
        combined.to_csv(FAIR_TABLE_DIR / '03_2_all_sources_bootstrap_summary.csv', index=False)
        display(combined.pivot_table(index='metric', columns='source', values='mean'))
else:
    print('Fair evaluator ready. Add optional prediction folders manually, then set RUN_FAIR_EVAL_NOW=True.')

## Paired bootstrap helper

Run this after evaluating at least two prediction folders with the fair evaluator. It compares models on shared case IDs only.


In [ ]:
def paired_bootstrap_diff(per_case_df: pd.DataFrame, source_a: str, source_b: str, metric: str, n_boot: int = 2000, seed: int = 42) -> Dict:
    a = per_case_df[per_case_df['source'] == source_a][['case_id', metric]].rename(columns={metric: 'a'})
    b = per_case_df[per_case_df['source'] == source_b][['case_id', metric]].rename(columns={metric: 'b'})
    merged = a.merge(b, on='case_id', how='inner')
    vals = pd.to_numeric(merged['a'], errors='coerce') - pd.to_numeric(merged['b'], errors='coerce')
    vals = vals.replace([np.inf, -np.inf], np.nan).dropna().to_numpy(dtype=float)
    if len(vals) == 0:
        return {'source_a': source_a, 'source_b': source_b, 'metric': metric, 'mean_diff': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'n': 0}
    rng = np.random.default_rng(seed)
    boots = [float(rng.choice(vals, size=len(vals), replace=True).mean()) for _ in range(n_boot)]
    return {
        'source_a': source_a,
        'source_b': source_b,
        'metric': metric,
        'mean_diff': float(vals.mean()),
        'ci_low': float(np.percentile(boots, 2.5)),
        'ci_high': float(np.percentile(boots, 97.5)),
        'n': int(len(vals)),
    }

# Example after fair eval:
# combined_cases = pd.concat(all_cases, ignore_index=True)
# pd.DataFrame([
#     paired_bootstrap_diff(combined_cases, '03_2_selected_student', 'external_baseline_name', m)
#     for m in ['dice', 'hd95_mm', 'nsd_2mm', 'volume_abs_error_ml']
# ])